# 00 - Data Folder Overview

This notebook is a compact overview of the CSVs in `seeder_data/data`: what each file represents, how tables connect, and quick quality snapshots.

In [25]:
from pathlib import Path

import pandas as pd

print('Imports loaded')

Imports loaded


In [26]:
REPO_ROOT = Path.cwd() if (Path.cwd() / 'seeder_data').exists() else Path.cwd().parent
DATA_DIR = REPO_ROOT / 'seeder_data' / 'data'
csv_files = sorted(DATA_DIR.glob('*.csv'))

print(f'Repo root: {REPO_ROOT}')
print(f'Data directory: {DATA_DIR}')
print(f'CSV files found: {len(csv_files)}')

Repo root: /Users/yousefhammoudeh/ufc-elo-calculator
Data directory: /Users/yousefhammoudeh/ufc-elo-calculator/seeder_data/data
CSV files found: 18


In [27]:
catalog_rows = []
for path in csv_files:
    columns = pd.read_csv(path, nrows=0).columns.tolist()
    with path.open('r', encoding='utf-8', newline='') as f:
        rows = max(sum(1 for _ in f) - 1, 0)
    catalog_rows.append(
        {'table': path.name, 'rows': rows, 'columns': len(columns), 'sample_columns': ', '.join(columns[:6])}
    )

catalog = pd.DataFrame(catalog_rows).sort_values('rows', ascending=False).reset_index(drop=True)
catalog

,table,rows,columns,sample_columns
0,tapology_fighter_urls_all.csv,469656,1,fighter_url
1,tapology_bout_fighters.csv,92638,9,"tapology_bout_id, tapology_fighter_id, result,..."
2,tapology_bouts.csv,72980,16,"tapology_bout_id, sport, is_amateur, is_title_..."
3,ufcstats_sig_strikes_by_position.csv,70194,13,"ufcstats_fight_id, ufcstats_fighter_id, round,..."
4,ufcstats_sig_strikes_by_target.csv,70194,13,"ufcstats_fight_id, ufcstats_fighter_id, round,..."
5,ufcstats_fight_rounds.csv,48664,17,"ufcstats_fight_id, ufcstats_fighter_id, round,..."
6,ufcstats_fight_totals.csv,21530,16,"ufcstats_fight_id, ufcstats_fighter_id, kd, si..."
7,tapology_events.csv,19667,8,"event_tapology_id, event_tapology_url, event_n..."
8,ufcstats_fights.csv,11093,16,"ufcstats_fight_id, ufcstats_fight_url, ufcstat..."
9,bout_id_map.csv,11093,2,"ufcstats_fight_id, tapology_bout_id"


## What Each CSV Means

### Tapology CSVs
- `tapology_fighter_urls_all.csv`: universe of fighter profile URLs discovered from Tapology.
- `tapology_fighters.csv`: one row per fighter with profile fields (name, DOB, size, stance, gym, location).
- `tapology_events.csv`: one row per event with date, promotion, and Tapology event ID.
- `tapology_bouts.csv`: one row per bout with method, weight class, event ID, promotion ID, and sport flags.
- `tapology_bout_fighters.csv`: one row per fighter per bout (result, opponent info, pre-fight records, source metadata).

### UFCStats CSVs
- `ufcstats_events.csv`: UFCStats events (date, location, event name).
- `ufcstats_events_2025.csv`: update subset focused on 2025 events.
- `ufcstats_fighters.csv`: UFCStats fighter profiles.
- `ufcstats_fights.csv`: fight-level outcome and metadata (winner, method, rounds, referee).
- `ufcstats_fight_totals.csv`: per-fighter totals for each fight.
- `ufcstats_fight_rounds.csv`: per-fighter stats by round.
- `ufcstats_sig_strikes_by_target.csv`: sig strikes split by head/body/leg.
- `ufcstats_sig_strikes_by_position.csv`: sig strikes split by distance/clinch/ground.

### Mapping and Metadata CSVs
- `fighter_id_map.csv`: links `ufcstats_fighter_id` to `tapology_fighter_id`.
- `bout_id_map.csv`: links `ufcstats_fight_id` to `tapology_bout_id`.
- `promotions_with_sports.csv`: cleaned promotion table with sport type and baseline strength.
- `promotions_full.csv`: larger promotion reference table with strength metadata.
- `sport_translation.csv`: lookup table for sport label normalization.


## Tapology Snapshot

In [28]:
tapology_tables = ['tapology_fighters.csv', 'tapology_events.csv', 'tapology_bouts.csv', 'tapology_bout_fighters.csv']
tapology_rows = []
for name in tapology_tables:
    df = pd.read_csv(DATA_DIR / name, dtype=str, low_memory=False)
    date_col = 'event_date' if 'event_date' in df.columns else None
    parsed = (
        pd.to_datetime(df[date_col].astype(str).str.replace(r'\\s+', ' ', regex=True).str.strip(), errors='coerce')
        if date_col
        else pd.Series(dtype='datetime64[ns]')
    )
    tapology_rows.append(
        {
            'table': name.replace('.csv', ''),
            'rows': len(df),
            'columns': df.shape[1],
            'date_min': parsed.min(),
            'date_max': parsed.max(),
        }
    )

pd.DataFrame(tapology_rows)

,table,rows,columns,date_min,date_max
0,tapology_fighters,4047,18,NaT,NaT
1,tapology_events,19667,8,1983-01-01,2025-12-30
2,tapology_bouts,72980,16,1980-04-25,2025-12-30
3,tapology_bout_fighters,92638,9,NaT,NaT


## UFCStats Snapshot

In [29]:
ufcstats_tables = [
    'ufcstats_events.csv',
    'ufcstats_fighters.csv',
    'ufcstats_fights.csv',
    'ufcstats_fight_totals.csv',
    'ufcstats_fight_rounds.csv',
]
ufcstats_rows = []
for name in ufcstats_tables:
    df = pd.read_csv(DATA_DIR / name, dtype=str, low_memory=False)
    date_col = 'event_date' if 'event_date' in df.columns else None
    parsed = pd.to_datetime(df[date_col], errors='coerce') if date_col else pd.Series(dtype='datetime64[ns]')
    ufcstats_rows.append(
        {
            'table': name.replace('.csv', ''),
            'rows': len(df),
            'columns': df.shape[1],
            'date_min': parsed.min(),
            'date_max': parsed.max(),
        }
    )

pd.DataFrame(ufcstats_rows)

,table,rows,columns,date_min,date_max
0,ufcstats_events,1232,6,1993-11-12,2025-12-13
1,ufcstats_fighters,4046,9,NaT,NaT
2,ufcstats_fights,11093,16,NaT,NaT
3,ufcstats_fight_totals,21530,16,NaT,NaT
4,ufcstats_fight_rounds,48664,17,NaT,NaT


## Mapping Coverage (How Tapology and UFCStats Connect)

In [30]:
fighter_map = pd.read_csv(DATA_DIR / 'fighter_id_map.csv', dtype=str)
bout_map = pd.read_csv(DATA_DIR / 'bout_id_map.csv', dtype=str)
tapology_fighters = pd.read_csv(DATA_DIR / 'tapology_fighters.csv', usecols=['tapology_fighter_id'], dtype=str)
tapology_bouts = pd.read_csv(DATA_DIR / 'tapology_bouts.csv', usecols=['tapology_bout_id'], dtype=str)
ufcstats_fighters = pd.read_csv(DATA_DIR / 'ufcstats_fighters.csv', usecols=['ufcstats_fighter_id'], dtype=str)
ufcstats_fights = pd.read_csv(DATA_DIR / 'ufcstats_fights.csv', usecols=['ufcstats_fight_id'], dtype=str)

coverage = pd.DataFrame(
    [
        {
            'entity': 'fighters',
            'mapped_rows': len(fighter_map),
            'tapology_total': tapology_fighters['tapology_fighter_id'].nunique(),
            'ufcstats_total': ufcstats_fighters['ufcstats_fighter_id'].nunique(),
        },
        {
            'entity': 'bouts/fights',
            'mapped_rows': len(bout_map),
            'tapology_total': tapology_bouts['tapology_bout_id'].nunique(),
            'ufcstats_total': ufcstats_fights['ufcstats_fight_id'].nunique(),
        },
    ]
)
coverage['mapped_vs_tapology_pct'] = (coverage['mapped_rows'] / coverage['tapology_total'] * 100).round(2)
coverage['mapped_vs_ufcstats_pct'] = (coverage['mapped_rows'] / coverage['ufcstats_total'] * 100).round(2)

print(f'fighter_id_map rows: {len(fighter_map):,}')
print(f'bout_id_map rows: {len(bout_map):,}')
coverage

fighter_id_map rows: 4,046
bout_id_map rows: 11,093


,entity,mapped_rows,tapology_total,ufcstats_total,mapped_vs_tapology_pct,mapped_vs_ufcstats_pct
0,fighters,4046,4047,4046,99.98,100.0
1,bouts/fights,11093,72980,11093,15.20,100.0


## Fighter Representation Example: Islam Makhachev

This section shows how one fighter appears across the linked CSVs: profile rows, mapping keys, Tapology bout rows, and linked UFCStats fights.

In [31]:
islam_name = 'Islam Makhachev'

tapology_islam = pd.read_csv(DATA_DIR / 'tapology_fighters.csv', dtype=str).query('name == @islam_name')
ufcstats_islam = pd.read_csv(DATA_DIR / 'ufcstats_fighters.csv', dtype=str).query('name == @islam_name')

print(f'Tapology matches: {len(tapology_islam)} | UFCStats matches: {len(ufcstats_islam)}')

Tapology matches: 1 | UFCStats matches: 1


In [32]:
tapology_islam[
    [
        'tapology_fighter_id',
        'name',
        'country',
        'dob',
        'fighting_out_of',
        'height_in',
        'reach_in',
        'stance',
        'foundation_style',
    ]
]

,tapology_fighter_id,name,country,dob,fighting_out_of,height_in,reach_in,stance,foundation_style
1456,40148-islam-makhachev,Islam Makhachev,RU,1991-10-27,"Makhachkala, Dagestan, Russia",70,70.5,NaN,NaN


In [33]:
ufcstats_islam[['ufcstats_fighter_id', 'name', 'record', 'dob', 'height_in', 'reach_in', 'stance']]

,ufcstats_fighter_id,name,record,dob,height_in,reach_in,stance
1537,275aca31f61ba28c,Islam Makhachev,28-1-0,1991-10-27,70.0,70.0,Southpaw


In [34]:
fighter_map_full = pd.read_csv(DATA_DIR / 'fighter_id_map.csv', dtype=str)

islam_map = fighter_map_full.merge(tapology_islam[['tapology_fighter_id']], on='tapology_fighter_id', how='inner')

islam_map

,ufcstats_fighter_id,tapology_fighter_id
0,275aca31f61ba28c,40148-islam-makhachev


In [35]:
islam_tapology_id = tapology_islam['tapology_fighter_id'].iloc[0]

islam_bout_rows = pd.read_csv(DATA_DIR / 'tapology_bout_fighters.csv', dtype=str)
islam_bout_rows = islam_bout_rows[islam_bout_rows['tapology_fighter_id'] == islam_tapology_id]

print(f'Tapology bout-fighter rows for Islam: {len(islam_bout_rows)}')
islam_bout_rows['result'].fillna('MISSING').value_counts().rename_axis('result').reset_index(name='rows')

Tapology bout-fighter rows for Islam: 29


,result,rows
0,W,28
1,L,1


In [36]:
tapology_bouts_full = pd.read_csv(
    DATA_DIR / 'tapology_bouts.csv',
    dtype=str,
    usecols=['tapology_bout_id', 'event_date', 'event_name', 'method', 'weight_class'],
)

islam_bouts = islam_bout_rows.merge(tapology_bouts_full, on='tapology_bout_id', how='left')

islam_bouts[['tapology_bout_id', 'result', 'event_date', 'event_name', 'method', 'weight_class']].head(10)

,tapology_bout_id,result,event_date,event_name,method,weight_class
0,1037618,W,2025-11-15,UFC 322,DEC,Welterweight
1,956505,W,2025-01-18,UFC 311: Makhachev vs. Moicano,SUB,Lightweight
2,864972,W,2024-06-01,UFC 302: Makhachev vs. Poirier,SUB,Lightweight
3,804607,W,2023-10-21,UFC 294: Makhachev vs. Volkanovski 2,TKO,Lightweight
4,700965,W,2023-02-11,UFC 284: Makhachev vs. Volkanovski,DEC,Lightweight
5,673222,W,2022-10-22,UFC 280: Oliveira vs. Makhachev,SUB,Lightweight
6,636096,W,2022-02-26,UFC Fight Night,TKO,Catchweight
7,603061,W,2021-10-30,UFC 267: Błachowicz vs. Teixeira,SUB,Lightweight
8,570389,W,2021-07-17,UFC Fight Night,SUB,Lightweight
9,542242,W,2021-03-06,UFC 259: Błachowicz vs. Adesanya,SUB,Lightweight


In [37]:
bout_map_full = pd.read_csv(DATA_DIR / 'bout_id_map.csv', dtype=str)
ufcstats_fights_small = pd.read_csv(
    DATA_DIR / 'ufcstats_fights.csv',
    dtype=str,
    usecols=['ufcstats_fight_id', 'weight_class', 'winner_fighter_id', 'method', 'scheduled_rounds'],
)

islam_bout_links = bout_map_full[bout_map_full['tapology_bout_id'].isin(islam_bouts['tapology_bout_id'])]

print(f'Islam Tapology bouts mapped to UFCStats fights: {islam_bout_links["ufcstats_fight_id"].nunique()}')
islam_bout_links.merge(ufcstats_fights_small, on='ufcstats_fight_id', how='left').head(10)

Islam Tapology bouts mapped to UFCStats fights: 18


,ufcstats_fight_id,tapology_bout_id,weight_class,scheduled_rounds,winner_fighter_id,method
0,0ea435ddf02deb60,804607,UFC Lightweight Title Bout,5.0,275aca31f61ba28c,KO/TKO
1,103a7d8c4f17714b,419558,Lightweight Bout,3.0,275aca31f61ba28c,Decision - Unanimous
2,144b0c1dd6c909f3,267995,Lightweight Bout,3.0,275aca31f61ba28c,Decision - Unanimous
3,1a0031e2072f21b6,864972,UFC Lightweight Title Bout,5.0,275aca31f61ba28c,Submission
4,256894b49303537b,700965,UFC Lightweight Title Bout,5.0,275aca31f61ba28c,Decision - Unanimous
5,2f4d42e0b9696f71,636096,Catch Weight Bout,5.0,275aca31f61ba28c,KO/TKO
6,3c3da227fbe36a28,185506,Lightweight Bout,3.0,275aca31f61ba28c,Submission
7,523815acf1bb2420,673222,UFC Lightweight Title Bout,5.0,275aca31f61ba28c,Submission
8,5ad4c0eae7512878,603061,Lightweight Bout,3.0,275aca31f61ba28c,Submission
9,678db400f85ae15c,216369,Lightweight Bout,3.0,7dcf06c1967801c1,KO/TKO


## Missingness Quick Check

In [38]:
missing_checks = {
    'tapology_bouts.csv': ['method', 'weight_class', 'event_date', 'promotion_tapology_id'],
    'tapology_bout_fighters.csv': ['result', 'opponent_tapology_slug', 'fighter_record_before_raw'],
    'tapology_fighters.csv': ['dob', 'height_in', 'reach_in', 'stance', 'foundation_style'],
    'ufcstats_fights.csv': ['winner_fighter_id', 'method', 'referee'],
}

missing_rows = []
for table, cols in missing_checks.items():
    df = pd.read_csv(DATA_DIR / table, usecols=cols)
    for col in cols:
        missing_rows.append({'table': table, 'column': col, 'missing_pct': round(df[col].isna().mean() * 100, 2)})

pd.DataFrame(missing_rows).sort_values(['table', 'missing_pct'], ascending=[True, False]).reset_index(drop=True)

,table,column,missing_pct
0,tapology_bout_fighters.csv,fighter_record_before_raw,9.08
1,tapology_bout_fighters.csv,opponent_tapology_slug,3.19
2,tapology_bout_fighters.csv,result,0.13
3,tapology_bouts.csv,weight_class,29.24
4,tapology_bouts.csv,promotion_tapology_id,15.76
5,tapology_bouts.csv,method,3.00
6,tapology_bouts.csv,event_date,0.00
7,tapology_fighters.csv,stance,100.00
8,tapology_fighters.csv,foundation_style,83.72
9,tapology_fighters.csv,reach_in,28.27


## Outcome and Method Distributions

In [39]:
tapology_bout_fighters = pd.read_csv(DATA_DIR / 'tapology_bout_fighters.csv', usecols=['result'])
tapology_bouts = pd.read_csv(DATA_DIR / 'tapology_bouts.csv', usecols=['method'])
ufcstats_fights = pd.read_csv(DATA_DIR / 'ufcstats_fights.csv', usecols=['method'])

tapology_result_dist = (
    tapology_bout_fighters['result'].fillna('MISSING').value_counts().rename_axis('result').reset_index(name='rows')
)
tapology_result_dist['pct'] = (tapology_result_dist['rows'] / tapology_result_dist['rows'].sum() * 100).round(2)

tapology_method_dist = (
    tapology_bouts['method'].fillna('MISSING').value_counts().head(12).rename_axis('method').reset_index(name='rows')
)
tapology_method_dist['pct'] = (tapology_method_dist['rows'] / len(tapology_bouts) * 100).round(2)

ufcstats_method_dist = (
    ufcstats_fights['method'].fillna('MISSING').value_counts().head(12).rename_axis('method').reset_index(name='rows')
)
ufcstats_method_dist['pct'] = (ufcstats_method_dist['rows'] / len(ufcstats_fights) * 100).round(2)

print('Tapology fighter-row outcomes')
print(tapology_result_dist.to_string(index=False))
print('\nTop Tapology bout methods')
print(tapology_method_dist.to_string(index=False))
print('\nTop UFCStats fight methods')
print(ufcstats_method_dist.to_string(index=False))

Tapology fighter-row outcomes
 result  rows   pct
      W 60264 65.05
      L 30090 32.48
      D  1456  1.57
     NC   703  0.76
MISSING   125  0.13

Top Tapology bout methods
 method  rows   pct
    TKO 25989 35.61
    SUB 22718 31.13
    DEC 21772 29.83
MISSING  2188  3.00
     DQ   313  0.43

Top UFCStats fight methods
                 method  rows   pct
   Decision - Unanimous  3734 33.66
                 KO/TKO  3591 32.37
             Submission  2346 21.15
       Decision - Split   992  8.94
TKO - Doctor's Stoppage   151  1.36
    Decision - Majority   110  0.99
             Overturned    70  0.63
     Could Not Continue    47  0.42
                     DQ    35  0.32
                  Other    14  0.13
               Decision     3  0.03


This notebook intentionally stays read-only and only reports what is already in `seeder_data/data`.